## Market Regimes

We will attempt to identify market states using supervised learning. I often find when I'm trading that markets shift very frequently. One week I can only extract a max of 1 or 2 Rs from a scalp trade and then another week the market is giving 3R or 4R trades. 

In [86]:
# import libraries
import numpy as np
import pandas as pd
import joblib
from math import log2
from sklearn.preprocessing import RobustScaler
from sklearn.mixture import GaussianMixture

In [87]:
# define csv file
rth_df = pd.read_csv("multiasset_5m_rth_features.csv")

print("Shape:", rth_df.shape)
print("Columns:")
print(rth_df.columns.tolist())

Shape: (171717, 27)
Columns:
['symbol', 'timestamp', 'ts_ny', 'date_ny', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'ret_1', 'ret_3', 'ret_6', 'ret_12', 'atr_14', 'atr_pct_14', 'vwap_session', 'vwap_dist', 'vwap_dist_atr', 'vol_ratio_20', 'range_pct', 'body_pct', 'upper_wick_pct', 'lower_wick_pct', 'rv_12', 'tod_sin', 'tod_cos']


In [88]:
# ensure timestamps account for DST
rth_df["ts_ny"] = pd.to_datetime(rth_df["ts_ny"], utc=True)
rth_df.head()

,symbol,timestamp,ts_ny,date_ny,open,high,low,close,volume,trade_count,...,vwap_dist,vwap_dist_atr,vol_ratio_20,range_pct,body_pct,upper_wick_pct,lower_wick_pct,rv_12,tod_sin,tod_cos
0,COIN,2024-02-28 18:35:00+00:00,2024-02-28 18:35:00+00:00,2024-02-28,201.26,202.930,201.1400,201.8700,210678.0,2749.0,...,-0.012738,-0.957690,0.531252,0.008867,0.003022,0.005251,0.000594,0.007565,-0.721202,-0.692724
1,COIN,2024-02-28 18:40:00+00:00,2024-02-28 18:40:00+00:00,2024-02-28,201.86,203.280,201.6800,202.1871,183109.0,2558.0,...,-0.010920,-0.827792,0.476845,0.007913,0.001618,0.005405,0.000890,0.007084,-0.774605,-0.632445
2,COIN,2024-02-28 18:45:00+00:00,2024-02-28 18:45:00+00:00,2024-02-28,202.30,203.820,201.8701,203.6350,280522.0,3099.0,...,-0.003521,-0.291903,0.726583,0.009575,0.006556,0.000908,0.002111,0.007378,-0.822984,-0.568065
3,COIN,2024-02-28 18:50:00+00:00,2024-02-28 18:50:00+00:00,2024-02-28,203.64,204.000,201.8602,203.6400,156113.0,2210.0,...,-0.003390,-0.289911,0.409781,0.010508,0.000000,0.001768,0.008740,0.007164,-0.866025,-0.500000
4,COIN,2024-02-28 18:55:00+00:00,2024-02-28 18:55:00+00:00,2024-02-28,203.77,205.585,203.3413,205.5189,232314.0,3074.0,...,0.005721,0.508484,0.614974,0.010917,0.008510,0.000322,0.002086,0.007683,-0.903450,-0.428693


In [89]:
# sort by stock and time stamp
rth_df = rth_df.sort_values(["ts_ny", "symbol"]).reset_index(drop=True)

print("Columns count:", len(rth_df.columns))
print("Symbols:", rth_df["symbol"].unique())
print("Date range:", rth_df["ts_ny"].min(), "to", rth_df["ts_ny"].max())

Columns count: 27
Symbols: ['TSLA' 'NVDA' 'PLTR' 'COIN' 'OKLO']
Date range: 2024-02-28 18:25:00+00:00 to 2026-02-27 16:55:00+00:00


In [90]:
# check for missing features
feature_cols = [
    "ret_1","ret_3","ret_6","ret_12",
    "atr_14","atr_pct_14",
    "vwap_session","vwap_dist","vwap_dist_atr",
    "vol_ratio_20",
    "range_pct","body_pct","upper_wick_pct","lower_wick_pct",
    "rv_12",
    "tod_sin","tod_cos"
]

missing_cols = [c for c in feature_cols if c not in rth_df.columns]
print("Missing feature columns:", missing_cols)

Missing feature columns: []


In [91]:
# validate price data is in chronological order
per_symbol_sorted = rth_df.groupby("symbol")["ts_ny"].apply(lambda s: s.is_monotonic_increasing)
print("\nEach Stock is in Chrono Order?")
print(per_symbol_sorted)


Each Stock is in Chrono Order?
symbol
COIN    True
NVDA    True
OKLO    True
PLTR    True
TSLA    True
Name: ts_ny, dtype: bool


In [92]:
# drop rows with NaNs features
mask_valid = rth_df[feature_cols].notna().all(axis=1)
print("Valid rows for clustering:", mask_valid.sum(), "/", len(rth_df))
print("Dropped rows due to NaNs:", (~mask_valid).sum())

Valid rows for clustering: 171717 / 171717
Dropped rows due to NaNs: 0


In [93]:
# verify price data range and daily trading sessions
rth_df["ts_ny"] = pd.to_datetime(rth_df["ts_ny"], utc=True)

rth_df["ts_local"] = rth_df["ts_ny"].dt.tz_convert("America/New_York")
rth_df["session_date"] = rth_df["ts_local"].dt.date

print("Calendar date range:", rth_df["session_date"].min(), "->", rth_df["session_date"].max())
print("Unique trading sessions:", rth_df["session_date"].nunique())

Calendar date range: 2024-02-28 -> 2026-02-27
Unique trading sessions: 484


In [94]:
# calculate daily summary features

# ensure stocks are sorted by time stamp
rth_df = rth_df.sort_values(["symbol", "ts_ny"]).reset_index(drop=True)

# calculate return from close to close for each stock
rth_df["ret_cc"] = rth_df.groupby("symbol")["close"].pct_change()

# calculate daily price data for each stock
daily = (
    rth_df
    .groupby(["symbol", "session_date"])
    .agg(
        day_open=("open","first"),
        day_close=("close","last"),
        day_high=("high","max"),
        day_low=("low","min"),
        day_volume=("volume","sum"),
        bars_in_day=("close","size"),
        daily_rv=("ret_cc", lambda x: np.nanstd(x, ddof=0)),
    )
    .reset_index()
)

daily["daily_ret"] = daily["day_close"] / daily["day_open"] - 1.0
daily["daily_range_pct"] = (daily["day_high"] - daily["day_low"]) / daily["day_open"]
daily["daily_trend_strength"] = daily["daily_ret"].abs() / (daily["daily_range_pct"] + 1e-12)

print("Daily table shape:", daily.shape)
print("\nSample rows:")
print(daily.head())

Daily table shape: (2237, 12)

Sample rows:
  symbol session_date  day_open  day_close  day_high   day_low  day_volume  \
0   COIN   2024-02-28    201.26    200.730   206.045  199.2400   4737289.0   
1   COIN   2024-02-29    206.46    203.420   211.310  193.8800  14412653.0   
2   COIN   2024-03-01    202.70    205.830   206.390  196.0101   8514752.0   
3   COIN   2024-03-04    217.40    228.720   236.460  212.2469  20685818.0   
4   COIN   2024-03-05    230.00    216.774   239.980  215.4000  21519305.0   

   bars_in_day  daily_rv  daily_ret  daily_range_pct  daily_trend_strength  
0           29  0.004005  -0.002633         0.033812              0.077884  
1           78  0.007627  -0.014724         0.084423              0.174412  
2           78  0.003980   0.015442         0.051208              0.301544  
3           78  0.008353   0.052070         0.111376              0.467516  
4           78  0.007343  -0.057504         0.106870              0.538080  


### Unsupervised Clustering

We want to mimic how I would actually trade with regimes. I would expect a regime to persist until price action shows me otherwise. 

Conceptually, if the past few trading sessions were choppy, I'd expect the current session to be choppy. If the past several sessions were trending, I'd expect the current trading session to offer trend.

In [95]:
# Create rolling macro features
daily = daily.sort_values(["symbol","session_date"]).reset_index(drop=True)

# using 5 day rolling period
N = 5 

# realized volatility
daily["rv_5"] = (
    daily.groupby("symbol")["daily_rv"]
    .shift(1)
    .rolling(N)
    .mean()
)

# range
daily["range_5"] = (
    daily.groupby("symbol")["daily_range_pct"]
    .shift(1)
    .rolling(N)
    .mean()
)

# trend
daily["trend_5"] = (
    daily.groupby("symbol")["daily_trend_strength"]
    .shift(1)
    .rolling(N)
    .mean()
)

# volume
daily["volume_5"] = (
    daily.groupby("symbol")["day_volume"]
    .shift(1)
    .rolling(N)
    .mean()
)

macro_feature_cols = [
    "rv_5",
    "range_5",
    "trend_5",
    "volume_5"
]

print(daily.head(10))

  symbol session_date  day_open  day_close  day_high   day_low  day_volume  \
0   COIN   2024-02-28    201.26   200.7300  206.0450  199.2400   4737289.0   
1   COIN   2024-02-29    206.46   203.4200  211.3100  193.8800  14412653.0   
2   COIN   2024-03-01    202.70   205.8300  206.3900  196.0101   8514752.0   
3   COIN   2024-03-04    217.40   228.7200  236.4600  212.2469  20685818.0   
4   COIN   2024-03-05    230.00   216.7740  239.9800  215.4000  21519305.0   
5   COIN   2024-03-06    229.20   238.7000  239.9000  223.0300  15701050.0   
6   COIN   2024-03-07    240.07   242.6200  242.8700  235.5000   9482267.0   
7   COIN   2024-03-08    246.00   256.7300  270.5476  244.9000  20278177.0   
8   COIN   2024-03-11    270.01   254.2900  271.6500  253.9600  17897955.0   
9   COIN   2024-03-12    257.65   256.2705  260.7900  242.0900  13142343.0   

   bars_in_day  daily_rv  daily_ret  daily_range_pct  daily_trend_strength  \
0           29  0.004005  -0.002633         0.033812           

In [96]:
# drop NaNs
macro_feature_cols = ["rv_5", "range_5", "trend_5", "volume_5"]
macro = daily.dropna(subset=macro_feature_cols).copy()

print(macro.head())

  symbol session_date  day_open  day_close  day_high  day_low  day_volume  \
5   COIN   2024-03-06    229.20   238.7000  239.9000   223.03  15701050.0   
6   COIN   2024-03-07    240.07   242.6200  242.8700   235.50   9482267.0   
7   COIN   2024-03-08    246.00   256.7300  270.5476   244.90  20278177.0   
8   COIN   2024-03-11    270.01   254.2900  271.6500   253.96  17897955.0   
9   COIN   2024-03-12    257.65   256.2705  260.7900   242.09  13142343.0   

   bars_in_day  daily_rv  daily_ret  daily_range_pct  daily_trend_strength  \
5           78  0.010300   0.041449         0.073604              0.563130   
6           78  0.003519   0.010622         0.030699              0.345997   
7           78  0.007894   0.043618         0.104259              0.418363   
8           78  0.006841  -0.058220         0.065516              0.888638   
9           78  0.005862  -0.005354         0.072579              0.073770   

       rv_5   range_5   trend_5    volume_5  
5  0.006262  0.077538 

In [97]:
entries = pd.read_csv("multiasset_labeled_entries.csv")
entries["ts_entry"] = pd.to_datetime(entries["ts_entry"], errors="coerce")
entries["ts_entry"] = entries["ts_entry"].dt.tz_localize("America/New_York").dt.tz_convert("UTC")

min_val_ts  = entries.loc[entries["split"]=="val", "ts_entry"].min()
min_test_ts = entries.loc[entries["split"]=="test", "ts_entry"].min()

first_val_day  = min_val_ts.tz_convert("America/New_York").date()
first_test_day = min_test_ts.tz_convert("America/New_York").date()

print("first_val_day :", first_val_day)
print("first_test_day:", first_test_day)

first_val_day : 2025-07-17
first_test_day: 2025-10-29


In [98]:
# split macro data
macro_train = macro[macro["session_date"] < first_val_day].copy()
macro_val   = macro[(macro["session_date"] >= first_val_day) & (macro["session_date"] < first_test_day)].copy()
macro_test  = macro[macro["session_date"] >= first_test_day].copy()

print("Macro train rows:", len(macro_train))
print("Macro val rows  :", len(macro_val))
print("Macro test rows :", len(macro_test))

Macro train rows: 1534
Macro val rows  : 330
Macro test rows : 348


In [99]:
# scale macro features
X_train_raw = macro_train[macro_feature_cols].astype(float).values
X_val_raw   = macro_val[macro_feature_cols].astype(float).values
X_test_raw  = macro_test[macro_feature_cols].astype(float).values

# only fit training data
scaler_macro = RobustScaler()
X_train = scaler_macro.fit_transform(X_train_raw)
X_val   = scaler_macro.transform(X_val_raw)
X_test  = scaler_macro.transform(X_test_raw)

print(X_train.shape, X_val.shape, X_test.shape)

(1534, 4) (330, 4) (348, 4)


For clustering, we'll go with Gaussian Mixture Model.

https://scikit-learn.org/stable/modules/mixture.html

Stock regimes overlap so GMM will be better to use.

In [100]:
# GMM on training data
Ks = range(2, 6)
for K in Ks:
    gmm = GaussianMixture(
        n_components=K,
        covariance_type="full",
        random_state=42,
        n_init=10,
        max_iter=800
    )
    gmm.fit(X_train)

    bic = gmm.bic(X_train)
    labels = gmm.predict(X_train)
    counts = np.bincount(labels, minlength=K)
    fracs = counts / counts.sum()

    print(f"K={K}  BIC={bic:,.0f}  min_frac={fracs.min():.3f}  max_frac={fracs.max():.3f}")

C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Window

K=2  BIC=12,513  min_frac=0.216  max_frac=0.784


C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Window

K=3  BIC=11,592  min_frac=0.143  max_frac=0.683


C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Window

K=4  BIC=9,694  min_frac=0.096  max_frac=0.379


C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Window

K=5  BIC=9,457  min_frac=0.064  max_frac=0.388


C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(


We'll go with K=4

K=5, BIC decreases but at relatively small compared to K=3 to K=4

In [101]:
# fit GMM
K = 4
gmm_macro = GaussianMixture(
    n_components=K,
    covariance_type="full",
    random_state=100,
    n_init=20,
    max_iter=1000
)
gmm_macro.fit(X_train)

C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=6.
  warnings.warn(
C:\Users\Donva\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Window

GaussianMixture(max_iter=1000, n_components=4, n_init=20, random_state=100)

In [102]:
# predict regimes for all data
X_all_raw = macro[macro_feature_cols].astype(float).values
X_all = scaler_macro.transform(X_all_raw)

probs = gmm_macro.predict_proba(X_all)
macro["macro_regime_id"] = probs.argmax(axis=1)
macro["macro_regime_pmax"] = probs.max(axis=1)

print("Macro regime distribution:")
print(macro["macro_regime_id"].value_counts(normalize=True).sort_index())

print("\nMacro confidence summary:")
print(macro["macro_regime_pmax"].describe())

Macro regime distribution:
macro_regime_id
0    0.156872
1    0.383363
2    0.375678
3    0.084087
Name: proportion, dtype: float64

Macro confidence summary:
count    2212.000000
mean        0.974136
std         0.071641
min         0.490618
25%         0.989848
50%         0.997608
75%         0.999102
max         1.000000
Name: macro_regime_pmax, dtype: float64


In [103]:
# add regime to 5M price data
rth_df = rth_df.merge(
    macro[["symbol", "session_date", "macro_regime_id", "macro_regime_pmax"]],
    on=["symbol", "session_date"],
    how="left"
)

print("Missing macro_regime_id on bars:", rth_df["macro_regime_id"].isna().sum())
print("Bars shape:", rth_df.shape)
print(rth_df[["symbol","ts_ny","session_date","macro_regime_id","macro_regime_pmax"]].head(10))

Missing macro_regime_id on bars: 1526
Bars shape: (171717, 32)
  symbol                     ts_ny session_date  macro_regime_id  \
0   COIN 2024-02-28 18:35:00+00:00   2024-02-28              NaN   
1   COIN 2024-02-28 18:40:00+00:00   2024-02-28              NaN   
2   COIN 2024-02-28 18:45:00+00:00   2024-02-28              NaN   
3   COIN 2024-02-28 18:50:00+00:00   2024-02-28              NaN   
4   COIN 2024-02-28 18:55:00+00:00   2024-02-28              NaN   
5   COIN 2024-02-28 19:00:00+00:00   2024-02-28              NaN   
6   COIN 2024-02-28 19:05:00+00:00   2024-02-28              NaN   
7   COIN 2024-02-28 19:10:00+00:00   2024-02-28              NaN   
8   COIN 2024-02-28 19:15:00+00:00   2024-02-28              NaN   
9   COIN 2024-02-28 19:20:00+00:00   2024-02-28              NaN   

   macro_regime_pmax  
0                NaN  
1                NaN  
2                NaN  
3                NaN  
4                NaN  
5                NaN  
6                NaN  
7   

In [106]:
# create regime dataset
rth_regime = rth_df.dropna(subset=["macro_regime_id"]).copy()
rth_regime["macro_regime_id"] = rth_regime["macro_regime_id"].astype(int)

print("Regime bars:", rth_regime.shape)
print("Dropped bars:", len(rth_df) - len(rth_regime))

Regime bars: (170191, 32)
Dropped bars: 1526


In [107]:
# add regimes to VWAP Reclaim Long labeled entries csv
entries_df = pd.read_csv("multiasset_labeled_entries.csv")
entries_df["ts_entry"] = pd.to_datetime(entries_df["ts_entry"], errors="coerce")
entries_df["ts_entry"] = entries_df["ts_entry"].dt.tz_localize("America/New_York").dt.tz_convert("UTC")
entries_df["session_date"] = entries_df["ts_entry"].dt.tz_convert("America/New_York").dt.date

entries_df = entries_df.merge(
    macro[["symbol","session_date","macro_regime_id","macro_regime_pmax"]],
    on=["symbol","session_date"],
    how="left"
)

print("Entries:", len(entries_df))
print("Entries missing macro_regime_id:", entries_df["macro_regime_id"].isna().sum())

Entries: 1710
Entries missing macro_regime_id: 17


In [109]:
# create entries with regime dataset
entries_regime = entries_df.dropna(subset=["macro_regime_id"]).copy()
entries_regime["macro_regime_id"] = entries_regime["macro_regime_id"].astype(int)

print("Regime entries:", len(entries_regime))

Regime entries: 1693


In [110]:
# save new datasets
rth_regime.to_csv("multiasset_5m_rth_features_with_regime.csv", index=False)
entries_regime.to_csv("multiasset_labeled_entries_with_regime.csv", index=False)

Now to test Regimes on VWAP Reclaim Setups

In [117]:
opps = pd.read_csv("multiasset_labeled_entries_with_regime.csv")
opps["ts_entry"] = pd.to_datetime(opps["ts_entry"], errors="coerce")
opps["ts_entry"] = opps["ts_entry"].dt.tz_convert("UTC")

opps = opps.dropna(subset=["macro_regime_id"]).copy()
opps["macro_regime_id"] = opps["macro_regime_id"].astype(int)

print("Opps shape:", opps.shape)
print(opps["split"].value_counts())

Opps shape: (1693, 34)
split
train    1180
test      257
val       256
Name: count, dtype: int64


best_R is from the simulated results of the VWAP Reclaim setups.

best_R = maximum simulated R-multiple achievable for that setup.

If best_R > 0, at least one of stop/target/hold combination resulted in a profitable trade.

In [121]:
# Win = best_R > 0
# calculate metrics for each regime
opps["win"] = (opps["best_R"] > 0).astype(int)

def regime_perf(df):
    g = df.groupby("macro_regime_id").agg(
        n=("best_R","size"),
        win_rate=("win","mean"),
        avg_R=("best_R","mean"),
        med_R=("best_R","median"),
        p25_R=("best_R", lambda x: np.percentile(x, 25)),
        p75_R=("best_R", lambda x: np.percentile(x, 75)),
    )
    g["expectancy"] = g["avg_R"]
    return g.sort_index()

train_perf = regime_perf(opps[opps["split"]=="train"])
val_perf   = regime_perf(opps[opps["split"]=="val"])
test_perf  = regime_perf(opps[opps["split"]=="test"])

print("\nTrain:")
print(train_perf)

print("\nValidation:")
print(val_perf)

print("\nTest:")
print(test_perf)


Train:
                   n  win_rate     avg_R     med_R  p25_R     p75_R  \
macro_regime_id                                                       
0                162  0.537037  0.977213  1.000000   -1.0  2.666667   
1                464  0.547414  1.116209  1.193610   -1.0  3.000000   
2                434  0.548387  1.088585  1.333333   -1.0  3.000000   
3                120  0.591667  1.087284  1.333333   -1.0  2.666667   

                 expectancy  
macro_regime_id              
0                  0.977213  
1                  1.116209  
2                  1.088585  
3                  1.087284  

Validation:
                   n  win_rate     avg_R     med_R  p25_R     p75_R  \
macro_regime_id                                                       
0                 50  0.540000  0.931139  1.000000   -1.0  2.589193   
1                 93  0.505376  1.013757  0.150329   -1.0  4.000000   
2                101  0.603960  1.162195  1.500000   -1.0  2.666667   
3                

In [123]:
macro.groupby("macro_regime_id")[macro_feature_cols].mean()

,rv_5,range_5,trend_5,volume_5
macro_regime_id,,,,
0,0.003141,0.035640,0.447346,2.007736e+08
1,0.005804,0.072908,0.472075,9.075777e+06
2,0.003529,0.042811,0.455772,6.642681e+07
3,0.010572,0.093253,0.471856,7.205882e+07


In [124]:
pd.crosstab(macro["macro_regime_id"], macro["symbol"], normalize="index")

symbol,COIN,NVDA,OKLO,PLTR,TSLA
macro_regime_id,,,,,
0,0.000000,1.000000,0.000000,0.000000,0.000000
1,0.524764,0.000000,0.475236,0.000000,0.000000
2,0.000000,0.064982,0.000000,0.454874,0.480144
3,0.086022,0.145161,0.338710,0.279570,0.150538
